## 젝시믹스 공식몰 리뷰 데이터 전처리

### 환경 설정

In [26]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [27]:
# 데이터 불러오기
df = pd.read_csv('./check_data/xexymix_product_review.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507561 entries, 0 to 507560
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   collect_date       507561 non-null  str    
 1   platform           507561 non-null  str    
 2   review_id          507561 non-null  int64  
 3   product_id         507561 non-null  int64  
 4   product_name       507561 non-null  str    
 5   review_date        507561 non-null  str    
 6   year               507561 non-null  int64  
 7   month              507561 non-null  int64  
 8   content            507215 non-null  str    
 9   rating             507561 non-null  int64  
 10  helpful_count      507561 non-null  int64  
 11  has_image          507561 non-null  int64  
 12  purchase_option    474212 non-null  str    
 13  user_height        445359 non-null  float64
 14  user_weight        437266 non-null  float64
 15  user_height_group  445359 non-null  str    
 16  user_weight_g

In [28]:
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-20,자사몰,5032253,2070608,UV 쉴드 에어핏 페이스 마스크,2026-03-09,2026,3,친구 소개로 구매했어요 친구가 걷기운동할때 필요해서 이것저것 구매했는데 기장 편하게...,5,1,1,색상:화이트 / 사이즈:L,158.0,50.0,155~159cm,50~54kg,https://www.xexymix.com/shop/shopdetail.html?b...
1,2026-04-20,자사몰,4998316,2070608,UV 쉴드 에어핏 페이스 마스크,2026-01-26,2026,1,l사이즈 사고 원단도 좋고 운동시 호흡도 편한데 큰 느낌이라 작은사이즈 다시 주문했...,5,0,1,색상:블랙 / 사이즈:S/M,168.0,55.0,165~169cm,55~59kg,https://www.xexymix.com/shop/shopdetail.html?b...
2,2026-04-20,자사몰,4936161,2070608,UV 쉴드 에어핏 페이스 마스크,2025-12-14,2025,12,구매하여 처음 사용해봤는데 가성비 진짜 좋네요 바람도 잘 막아주고 추운날씨에도 딱 ...,5,0,1,색상:블랙 / 사이즈:L,179.0,70.0,180~184cm,70~74kg,https://www.xexymix.com/shop/shopdetail.html?b...
3,2026-04-20,자사몰,4727267,2070608,UV 쉴드 에어핏 페이스 마스크,2025-07-22,2025,7,전부터 풀페이스마스크 하나 사려고 보고 있다가 지난번 블랙색상 사고 마음에 들어서 ...,5,4,1,색상:화이트 / 사이즈:L,170.0,60.0,170~174cm,60~64kg,https://www.xexymix.com/shop/shopdetail.html?b...
4,2026-04-20,자사몰,4821424,2070608,UV 쉴드 에어핏 페이스 마스크,2025-10-10,2025,10,낮에 러닝하는데 얼굴 타는게 신경쓰였는데 마스크 답답하지 않고 흘러 내리지 않아 좋아요,5,1,1,색상:화이트 / 사이즈:S/M,153.0,45.0,150~154cm,45~49kg,https://www.xexymix.com/shop/shopdetail.html?b...


---
## purchase_option 컬럼 파생변수 생성

### purchase_option 원본 형식 파악

In [29]:
po = df['purchase_option'].dropna()

# ── 전체 패턴 분류 ──
def classify_pattern(s):
    if re.match(r'^선택\d+:', s):
        return '선택N 패턴'
    elif re.match(r'^.+\([A-Z0-9_]+\):', s):
        return '코드포함 세트상품'
    elif '색상:' in s and '사이즈:' in s:
        return '색상+사이즈 KV'
    elif '색상:' in s:
        return '색상만 KV'
    elif '사이즈:' in s or 'SIZE:' in s:
        return '사이즈만 KV'
    elif '기장:' in s:
        return '기장+α KV'
    elif '옵션:' in s:
        return '옵션 단독'
    elif '추가상품' in s:
        return '추가상품'
    else:
        return '기타'

pattern_counts = po.apply(classify_pattern).value_counts()
print("=== purchase_option 패턴 분포 ===")
print(pattern_counts.to_string())
print(f"\n총 non-null: {len(po):,}건 / 전체: {len(df):,}건")
print(f"null(구매옵션 없음): {df['purchase_option'].isna().sum():,}건")

=== purchase_option 패턴 분포 ===
purchase_option
색상+사이즈 KV    143965
선택N 패턴       135462
사이즈만 KV      108328
색상만 KV        67225
코드포함 세트상품     14038
옵션 단독          3289
추가상품           1891
기타               14

총 non-null: 474,212건 / 전체: 507,561건
null(구매옵션 없음): 33,349건


### 파싱 함수 정의

In [30]:
# ════════════════════════════════════════════════════════
#  사이즈 판별 정규식
#  우선순위 보장을 위해 긴 패턴 먼저 배치
# ════════════════════════════════════════════════════════
_SL  = r'(?:XXXL|XXL|2XL|3XL|4XL|XL|XS|4L|6L|S|M|L)'       # 기본 사이즈 심볼
_SC  = rf'{_SL}(?:[/\-]{_SL})?'                               # S/M, L-XL 등 복합
_SP  = rf'{_SC}(?:\s*\([^)]*\))*'                             # S(44~55), M (95), L(+2000) 등

SIZE_PATTERN = re.compile(
    rf'^(?:'
    rf'{_SP}'                       # 알파벳 계열 (S, M, L, XL, S/M, S(44~55), M(95) …)
    rf'|\d{{2,3}}'                  # 순수 숫자 (230, 240, 28, 29, 30 …)
    rf'|\d{{1,2}}호'                # 호 사이즈 (18호, 19호 …)
    rf'|\d{{2,3}}\([^)]+\)'        # 허리/힙 콤보 (82(L), 85(L-XL) …)
    rf'|대|소|프리사이즈|FREE'
    rf')$',
    re.IGNORECASE
)

# ════════════════════════════════════════════════════════
#  여성사이즈 정확 매핑
#  반드시 아래 7가지 패턴과 정확히 일치해야만 치환
#  S(90), M(95), L(100), XL(105) 등 공용사이즈는 포함되지 않음
# ════════════════════════════════════════════════════════
WOMEN_SIZE_MAP = {
    'XS(80)':   'WXS(80)',
    'S(85)':    'WS(85)',
    'M(90)':    'WM(90)',
    'L(95)':    'WL(95)',
    'XL(100)':  'WXL(100)',
    '2XL(105)': 'W2XL(105)',
    '3XL(110)': 'W3XL(110)',
}

# ════════════════════════════════════════════════════════
#  유틸 함수
# ════════════════════════════════════════════════════════
def normalize_size(s: str) -> str:
    """가격 suffix 제거 및 공백 정규화"""
    s = s.strip()
    s = re.sub(r'\s*\(\+\d+\)', '', s)   # (+2000), (+27395) 제거
    s = re.sub(r'(\w)\s+\(', r'\1(', s)  # 'M (95)' → 'M(95)'
    return s.strip()

def is_size_token(token: str) -> bool:
    return bool(SIZE_PATTERN.match(token.strip()))

def apply_women_size(size_str: str):
    """
    여성사이즈 정확 치환
    XS(80), S(85), M(90), L(95), XL(100), 2XL(105), 3XL(110) 만 치환
    S(90), M(95), L(100), XL(105) 등 공용사이즈는 치환 안 함
    Returns: (치환된_문자열, is_women: bool)
    """
    normalized = size_str.strip()
    if normalized in WOMEN_SIZE_MAP:
        return WOMEN_SIZE_MAP[normalized], True
    return size_str, False

# ════════════════════════════════════════════════════════
#  메인 파싱 함수
# ════════════════════════════════════════════════════════
def parse_purchase_option(option_str):
    """
    purchase_option 문자열에서 색상 목록과 사이즈 목록을 추출

    지원 패턴:
      1. 색상:XXX / 사이즈:YYY          (표준 KV)
      2. 사이즈:YYY / SIZE:YYY          (사이즈만 KV)
      3. 기장:N / 색상:XXX / 사이즈:YYY  (부가정보 포함 KV)
      4. 선택N:COLOR SIZE              (다중 선택)
      5. 선택N:PREFIX COLOR SIZE       (스타일 접두어 포함)
      6. 선택N:COLOR                  (사이즈 없는 다중 선택)
      7. 상의(CODE):COLOR SIZE / 하의(CODE):COLOR SIZE  (코드 포함 세트)
      8. 기타 (옵션, 추가상품 등) → 스킵

    Returns: (colors: list, sizes: list)
    """
    if pd.isna(option_str):
        return [], []

    colors, sizes = [], []

    for part in [p.strip() for p in str(option_str).split(' / ')]:
        colon_idx = part.find(':')
        if colon_idx == -1:
            continue

        key    = part[:colon_idx].strip()
        value  = part[colon_idx + 1:].strip()
        tokens = value.split()

        # ── 패턴 1: 색상 키 ──
        if key == '색상':
            if value:
                colors.append(value)

        # ── 패턴 2: 사이즈 키 (한글/영문) ──
        elif key in ('사이즈', 'SIZE'):
            if not tokens:
                continue
            # '사이즈:블랙 S' 처럼 색상+사이즈가 합쳐진 경우 분리
            norm_last = normalize_size(tokens[-1])
            if len(tokens) >= 2 and is_size_token(norm_last):
                sizes.append(norm_last)
                for t in reversed(tokens[:-1]):
                    if re.search(r'[가-힣]', t):
                        colors.append(t)
                        break
            else:
                sizes.append(normalize_size(value))

        # ── 패턴 3: 선택N / 증정 ──
        elif re.match(r'^(?:선택\d+|증정\d*)', key):
            if not tokens:
                continue
            norm_last = normalize_size(tokens[-1])
            if is_size_token(norm_last):
                sizes.append(norm_last)
                for t in reversed(tokens[:-1]):
                    if re.search(r'[가-힣]', t):
                        colors.append(t)
                        break
            else:
                # 마지막 한글 토큰 = 색상
                for t in reversed(tokens):
                    if re.search(r'[가-힣]', t):
                        colors.append(t)
                        break

        # ── 패턴 4: 코드 포함 세트상품 키  예) 상의(CODE), 집업(CODE) ──
        elif re.match(r'^.+\(.+\)$', key):
            if not tokens:
                continue
            norm_last = normalize_size(tokens[-1])
            if is_size_token(norm_last):
                sizes.append(norm_last)
                for t in reversed(tokens[:-1]):
                    if re.search(r'[가-힣]', t):
                        colors.append(t)
                        break
            else:
                for t in reversed(tokens):
                    if re.search(r'[가-힣]', t):
                        colors.append(t)
                        break

        # ── 기타 키 (기장, 소재, 옵션, 추가상품, 기본패드 등) → 스킵 ──

    return colors, sizes

print("파싱 함수 정의 완료")

파싱 함수 정의 완료


### 파싱 결과 검증 (샘플)

In [31]:
# 각 패턴별 대표 케이스로 파싱 검증
# 컬러: 중복 유지 + 정렬 / 사이즈: 중복 유지 + 정렬
test_cases = [
    # 표준 KV
    ('색상:화이트 / 사이즈:L',                               '화이트',           'L'),
    ('색상:블랙 / 사이즈:S/M',                               '블랙',             'S/M'),
    ('색상:스킨 / 사이즈:M(55반~66)',                         '스킨',             'M(55반~66)'),
    ('색상:블랙 / 사이즈:XXL',                               '블랙',             'XXL'),
    # 사이즈만
    ('사이즈:240',                                         '-',               '240'),
    ('사이즈:19호',                                        '-',               '19호'),
    ('사이즈:S(44~55)',                                    '-',               'S(44~55)'),
    ('사이즈:M(55반~66)',                                   '-',               'M(55반~66)'),
    # 가격 suffix 제거
    ('색상:블랙 / 사이즈:L(+2000)',                          '블랙',             'L'),
    ('색상:블랙 / 사이즈:S(44~55)(+2000)',                   '블랙',             'S(44~55)'),
    # 색상+사이즈 혼합
    ('사이즈:블랙 S',                                       '블랙',             'S'),
    # 공용사이즈 (치환 안 함)
    ('사이즈:M (95)',                                      '-',               'M(95)'),
    ('사이즈:S (90)',                                      '-',               'S(90)'),
    ('색상:스킨 / 사이즈:L(100)',                             '스킨',             'L(100)'),
    ('색상:블랙 / 사이즈:XL(105)',                           '블랙',             'XL(105)'),
    # 여성사이즈 (치환 함) - 정확히 7가지 패턴만
    ('사이즈:M(90)',                                       '-',               'WM(90)'),
    ('사이즈:L(95)',                                       '-',               'WL(95)'),
    ('사이즈:XL(100)',                                     '-',               'WXL(100)'),
    ('사이즈:2XL(105)',                                    '-',               'W2XL(105)'),
    # SIZE 영문 키
    ('SIZE:M(55반~66) / 기본패드:에어브라패드',                  '-',               'M(55반~66)'),
    # 기장 포함 KV
    ('기장:10부 / 색상:블랙 / 사이즈:M(55반~66)',               '블랙',             'M(55반~66)'),
    # 선택N (단색, 단일 사이즈)
    ('선택1:블랙 M / 선택2:백아이보리 M',                       '백아이보리,블랙',     'M,M'),
    # 선택N (동일 컬러+사이즈 반복 → 중복 유지)
    ('선택1:블랙 M / 선택2:블랙 M',                          '블랙,블랙',          'M,M'),
    # 선택N (멀티컬러, 사이즈 없음)
    ('선택1:미네랄민트 / 선택2:튤립핑크',                        '미네랄민트,튤립핑크',  '-'),
    # 선택N (스타일 접두어)
    ('선택1:숏슬리브 블랙 XL / 선택2:숏슬리브 더스틴네이비 XL',        '더스틴네이비,블랙',   'XL,XL'),
    ('선택1:기모 9.1 블랙 XL(+2000) / 선택2:기모 9.1 차콜 XL(+2000)', '블랙,차콜',     'XL,XL'),
    ('선택1:9부 블랙 M / 선택2:9부 세이지쿠키 M',                 '블랙,세이지쿠키',    'M,M'),
    # 선택N (사이즈 없음)
    ('선택1:요가삭스 블랙 / 선택2:요가삭스 스틸네이비',              '블랙,스틸네이비',    '-'),
    # 세트상품
    ('집입(XSFRZ20J2):아이보리 M / 레깅스(XSFWL20J2):빅토리아블루 M', '빅토리아블루,아이보리', 'M,M'),
    ('상의(XFK4JL1102):블랙 M / 하의(PS1102_PT1108):롱 블랙 M(+25000)', '블랙',      'M,M'),
    # 허리/힙 사이즈
    ('사이즈:82(L)',                                        '-',               '82(L)'),
    ('사이즈:85(L-XL)',                                     '-',               '85(L-XL)'),
]

# ── 여성사이즈 치환 적용 후 비교 ──
def apply_women_to_sizes(size_str):
    if not size_str or size_str == '-':
        return size_str
    parts = [sz.strip() for sz in size_str.split(',')]
    return ','.join(apply_women_size(sz)[0] for sz in parts)

print(f"{'purchase_option':<57} {'색상OK':<6} {'사이즈OK':<7} {'색상결과':<22} {'사이즈결과'}")
print('-' * 115)
fail = 0
for opt, exp_c, exp_s in test_cases:
    c, s = parse_purchase_option(opt)
    got_c = ','.join(sorted(c)) if c else '-'   # 중복 유지, 정렬
    got_s = ','.join(sorted(s)) if s else '-'   # 중복 유지, 정렬
    got_s_w = apply_women_to_sizes(got_s)       # 여성사이즈 치환
    c_ok = '✓' if got_c == exp_c else '✗'
    s_ok = '✓' if got_s_w == exp_s else '✗'
    if c_ok == '✗' or s_ok == '✗':
        fail += 1
    print(f"{opt:<57} {c_ok:<6} {s_ok:<7} {got_c:<22} {got_s_w}")

print(f"\n실패: {fail}/{len(test_cases)}")

print("\n=== 여성사이즈 치환 단독 테스트 ===")
print(f"{'입력':<15} {'결과':<20} {'치환여부'}")
for sz in ['XS(80)', 'S(85)', 'M(90)', 'L(95)', 'XL(100)', '2XL(105)', '3XL(110)',
           'S(90)', 'M(95)', 'L(100)', 'XL(105)', 'S', 'M', 'L', 'XL',
           'S(44~55)', 'M(55반~66)', 'XS(44)']:
    result, is_w = apply_women_size(sz)
    marker = '← 치환' if is_w else ''
    print(f"{sz:<15} {result:<20} {marker}")

purchase_option                                           색상OK   사이즈OK   색상결과                   사이즈결과
-------------------------------------------------------------------------------------------------------------------
색상:화이트 / 사이즈:L                                            ✓      ✓       화이트                    L
색상:블랙 / 사이즈:S/M                                           ✓      ✓       블랙                     S/M
색상:스킨 / 사이즈:M(55반~66)                                     ✓      ✓       스킨                     M(55반~66)
색상:블랙 / 사이즈:XXL                                           ✓      ✓       블랙                     XXL
사이즈:240                                                   ✓      ✓       -                      240
사이즈:19호                                                   ✓      ✓       -                      19호
사이즈:S(44~55)                                              ✓      ✓       -                      S(44~55)
사이즈:M(55반~66)                                             ✓      ✓       

### 파생 컬럼 생성: purchase_option_color / purchase_option_size / women_size_yn

In [32]:
print("purchase_option 파싱 중...")

parsed = df['purchase_option'].apply(parse_purchase_option)
colors_list = [r[0] for r in parsed]
sizes_list  = [r[1] for r in parsed]

# ── purchase_option_color: 멀티컬러 중복 유지, 정렬만 적용 ──
df['purchase_option_color'] = [
    ','.join(sorted(c)) if c else np.nan
    for c in colors_list
]

# ── purchase_option_size: 멀티사이즈 중복 유지, 정렬만 적용 ──
df['purchase_option_size'] = [
    ','.join(sorted(s)) if s else np.nan
    for s in sizes_list
]

# ── 여성사이즈 치환 및 women_size_yn 생성 ──
def apply_women_size_to_col(size_val):
    """purchase_option_size 셀 값에 여성사이즈 치환 적용"""
    if pd.isna(size_val):
        return size_val, 0

    parts     = [sz.strip() for sz in str(size_val).split(',')]
    new_parts = []
    any_women = False

    for sz in parts:
        replaced, is_women = apply_women_size(sz)
        new_parts.append(replaced)
        if is_women:
            any_women = True

    return ','.join(new_parts), int(any_women)

result = df['purchase_option_size'].apply(apply_women_size_to_col)
df['purchase_option_size'] = [r[0] for r in result]
df['women_size_yn']         = [r[1] for r in result]

print("완료!")
print(f"\n컬럼별 현황:")
print(f"  purchase_option_color : non-null {df['purchase_option_color'].notna().sum():,}건")
print(f"  purchase_option_size  : non-null {df['purchase_option_size'].notna().sum():,}건")
print(f"  women_size_yn=1       : {(df['women_size_yn']==1).sum():,}건")

print(f"\n신규 컬럼 확인:")
df[['purchase_option', 'purchase_option_color', 'purchase_option_size', 'women_size_yn']].head(10)

purchase_option 파싱 중...
완료!

컬럼별 현황:
  purchase_option_color : non-null 361,542건
  purchase_option_size  : non-null 389,474건
  women_size_yn=1       : 0건

신규 컬럼 확인:


,purchase_option,purchase_option_color,purchase_option_size,women_size_yn
0,색상:화이트 / 사이즈:L,화이트,L,0
1,색상:블랙 / 사이즈:S/M,블랙,S/M,0
2,색상:블랙 / 사이즈:L,블랙,L,0
3,색상:화이트 / 사이즈:L,화이트,L,0
4,색상:화이트 / 사이즈:S/M,화이트,S/M,0
5,색상:블랙 / 사이즈:L,블랙,L,0
6,색상:화이트 / 사이즈:L,화이트,L,0
7,색상:화이트 / 사이즈:S/M,화이트,S/M,0
8,색상:화이트 / 사이즈:L,화이트,L,0
9,색상:화이트 / 사이즈:S/M,화이트,S/M,0


### 결과 검증

In [33]:
# ── 사이즈 유니크 값 분포 ──
print("=== purchase_option_size 상위 50개 ===")
print(df['purchase_option_size'].value_counts().head(50).to_string())

print("\n=== 멀티 사이즈 케이스 (콤마 포함) ===")
multi_sz = df[df['purchase_option_size'].str.contains(',', na=False)]
print(f"건수: {len(multi_sz):,}건")
print(multi_sz['purchase_option_size'].value_counts().head(10).to_string())

=== purchase_option_size 상위 50개 ===
purchase_option_size
M(55반~66)         61629
S(44~55)          42596
M,M               34635
L(66반~77)         30744
L                 30322
L,L               25017
S,S               19187
XL                16010
XL,XL             14110
S/M               11661
M                 11197
XL(77~88)         10452
XXL                7382
240                5830
XXL,XXL            5379
L,L,L              4130
M,M,M              3708
230                3312
XXL(88반~99)        3108
L,M                3079
250                2908
S                  2762
M,S                2683
XL,XL,XL           2659
XXXL,XXXL          2593
XXXL               2441
L,XL               1658
S,S,S              1560
XXL,XXL,XXL        1238
245                1190
XS(44)             1137
235                1047
S-M,S-M,S-M         950
S-M                 905
S/M,S/M             897
M,M,M,M,M           857
270                 773
L,L,L,L,L           768
260                 745
XXXL(99

In [34]:
# ── 컬러 유니크 값 분포 ──
print("=== purchase_option_color 상위 30개 ===")
print(df['purchase_option_color'].value_counts().head(30).to_string())

print("\n=== 멀티 컬러 케이스 (콤마 포함) ===")
multi_col = df[df['purchase_option_color'].str.contains(',', na=False)]
print(f"건수: {len(multi_col):,}건")
print(multi_col['purchase_option_color'].value_counts().head(10).to_string())

=== purchase_option_color 상위 30개 ===
purchase_option_color
블랙           89895
화이트          20881
블랙,블랙        13975
백아이보리         6801
스킨            4986
오프화이트         4750
아이보리          4340
백아이보리,블랙      4331
멜란지그레이        2735
베이지           2635
차콜            2211
그린            2038
블랙,화이트        1793
네이비           1725
블루            1404
카키            1386
블랙,아이보리       1334
라이트그레이        1254
라이트블루         1236
라이트베이지        1236
핑크            1232
다크그레이,블랙      1195
멜란지그레이,블랙     1192
블랙,블랙,블랙      1170
네이비,블랙        1090
스카이블루         1078
블랙,차콜         1016
그레이            906
라이트핑크          892
크림아이보리         879

=== 멀티 컬러 케이스 (콤마 포함) ===
건수: 149,500건
purchase_option_color
블랙,블랙          13975
백아이보리,블랙        4331
블랙,화이트          1793
블랙,아이보리         1334
다크그레이,블랙        1195
멜란지그레이,블랙       1192
블랙,블랙,블랙        1170
네이비,블랙          1090
블랙,차콜           1016
백아이보리,백아이보리      762


In [35]:
# ── 여성사이즈 치환 결과 검증 ──
print("=== 여성사이즈 치환 전후 확인 ===")
women_rows = df[df['women_size_yn'] == 1][['purchase_option', 'purchase_option_size', 'women_size_yn']]
print(f"치환된 행 수: {len(women_rows):,}건\n")

print("치환된 size 분포:")
print(women_rows['purchase_option_size'].value_counts().to_string())

print("\n원본 vs 치환 샘플:")
print(women_rows[['purchase_option', 'purchase_option_size']].to_string())

=== 여성사이즈 치환 전후 확인 ===
치환된 행 수: 0건

치환된 size 분포:
Series([], )

원본 vs 치환 샘플:
Empty DataFrame
Columns: [purchase_option, purchase_option_size]
Index: []


In [36]:
# ── 공용사이즈 미치환 확인 (S, M, L 등 단독은 women_size_yn=0이어야 함) ──
print("=== 공용사이즈(S/M/L/XL) 치환 여부 확인 ===")
common_size_check = df[
    df['purchase_option_size'].isin(['S', 'M', 'L', 'XL', 'XXL', 'XXXL', 'XS'])
]['women_size_yn'].value_counts()
print("공용사이즈 women_size_yn 분포 (모두 0이어야 정상):")
print(common_size_check.to_string())

print("\n=== 최종 컬럼 요약 ===")
print(df[['purchase_option_color', 'purchase_option_size', 'women_size_yn']].describe(include='all'))

=== 공용사이즈(S/M/L/XL) 치환 여부 확인 ===
공용사이즈 women_size_yn 분포 (모두 0이어야 정상):
women_size_yn
0    70114

=== 최종 컬럼 요약 ===
       purchase_option_color purchase_option_size  women_size_yn
count                 361542               389474       507561.0
unique                 13196                  518            NaN
top                       블랙            M(55반~66)            NaN
freq                   89895                61629            NaN
mean                     NaN                  NaN            0.0
std                      NaN                  NaN            0.0
min                      NaN                  NaN            0.0
25%                      NaN                  NaN            0.0
50%                      NaN                  NaN            0.0
75%                      NaN                  NaN            0.0
max                      NaN                  NaN            0.0


In [37]:
# ========================
# 최종 컬럼 정리
# ========================

# 1) 데이터 형변환
def convert_dtypes(df):
    # 날짜 컬럼
    for col in ["collect_date", "review_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # ID 컬럼
    for col in ["review_id", "product_id"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # 문자열 컬럼
    for col in ["product_name", "content", "purchase_option", "product_url"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # category 컬럼
    for col in ["platform", "purchase_option_color", "purchase_option_size", "user_height_group", "user_weight_group"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # 정수형 컬럼
    for col, dtype in [("year", "Int16"), ("month", "Int8"), ("rating", "Int8"), ("helpful_count", "Int32")]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    # 0/1 플래그 컬럼
    for col in ["has_image", "women_size_yn"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    # 실수형 컬럼
    for col in ["user_height", "user_weight"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df

# 사용
df = convert_dtypes(df)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 507561 entries, 0 to 507560
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   collect_date           507561 non-null  datetime64[us]
 1   platform               507561 non-null  category      
 2   review_id              507561 non-null  string        
 3   product_id             507561 non-null  string        
 4   product_name           507561 non-null  string        
 5   review_date            507561 non-null  datetime64[us]
 6   year                   507561 non-null  Int16         
 7   month                  507561 non-null  Int8          
 8   content                507215 non-null  string        
 9   rating                 507561 non-null  Int8          
 10  helpful_count          507561 non-null  Int32         
 11  has_image              507561 non-null  int8          
 12  purchase_option        474212 non-null  string        


In [38]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택 (안전하게)
df = df[
    [col for col in column_order if col in df.columns]
]

# 확인
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-20,자사몰,5032253,2070608,UV 쉴드 에어핏 페이스 마스크,2026-03-09,2026,3,친구 소개로 구매했어요 친구가 걷기운동할때 필요해서 이것저것 구매했는데 기장 편하게...,5,1,1,색상:화이트 / 사이즈:L,화이트,L,0,158.0,50.0,155~159cm,50~54kg,https://www.xexymix.com/shop/shopdetail.html?b...
1,2026-04-20,자사몰,4998316,2070608,UV 쉴드 에어핏 페이스 마스크,2026-01-26,2026,1,l사이즈 사고 원단도 좋고 운동시 호흡도 편한데 큰 느낌이라 작은사이즈 다시 주문했...,5,0,1,색상:블랙 / 사이즈:S/M,블랙,S/M,0,168.0,55.0,165~169cm,55~59kg,https://www.xexymix.com/shop/shopdetail.html?b...
2,2026-04-20,자사몰,4936161,2070608,UV 쉴드 에어핏 페이스 마스크,2025-12-14,2025,12,구매하여 처음 사용해봤는데 가성비 진짜 좋네요 바람도 잘 막아주고 추운날씨에도 딱 ...,5,0,1,색상:블랙 / 사이즈:L,블랙,L,0,179.0,70.0,180~184cm,70~74kg,https://www.xexymix.com/shop/shopdetail.html?b...
3,2026-04-20,자사몰,4727267,2070608,UV 쉴드 에어핏 페이스 마스크,2025-07-22,2025,7,전부터 풀페이스마스크 하나 사려고 보고 있다가 지난번 블랙색상 사고 마음에 들어서 ...,5,4,1,색상:화이트 / 사이즈:L,화이트,L,0,170.0,60.0,170~174cm,60~64kg,https://www.xexymix.com/shop/shopdetail.html?b...
4,2026-04-20,자사몰,4821424,2070608,UV 쉴드 에어핏 페이스 마스크,2025-10-10,2025,10,낮에 러닝하는데 얼굴 타는게 신경쓰였는데 마스크 답답하지 않고 흘러 내리지 않아 좋아요,5,1,1,색상:화이트 / 사이즈:S/M,화이트,S/M,0,153.0,45.0,150~154cm,45~49kg,https://www.xexymix.com/shop/shopdetail.html?b...


In [39]:
# 띄어쓰기 제외 5글자 이하인 리뷰 제거
df['content_length'] = df['content'].str.replace(r'\s+', '', regex=True).str.len()
df = df[df['content_length'] > 5].drop(columns=['content_length'])
print(f"최종 리뷰 건수: {len(df):,}건")

최종 리뷰 건수: 494,159건


In [40]:
df.info()

<class 'pandas.DataFrame'>
Index: 494159 entries, 0 to 507560
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   collect_date           494159 non-null  datetime64[us]
 1   platform               494159 non-null  category      
 2   review_id              494159 non-null  string        
 3   product_id             494159 non-null  string        
 4   product_name           494159 non-null  string        
 5   review_date            494159 non-null  datetime64[us]
 6   year                   494159 non-null  Int16         
 7   month                  494159 non-null  Int8          
 8   content                494159 non-null  string        
 9   rating                 494159 non-null  Int8          
 10  helpful_count          494159 non-null  Int32         
 11  has_image              494159 non-null  int8          
 12  purchase_option        460810 non-null  string        
 13  

---
## 결측치 발생 원인 검증

In [41]:
total = len(df)

# ── 1. 컬럼별 결측치 요약 ──
null_summary = pd.DataFrame({
    '전체': total,
    'non-null': df.notna().sum(),
    '결측치': df.isna().sum(),
    '결측률(%)': (df.isna().sum() / total * 100).round(2)
})
print("=== 컬럼별 결측치 요약 ===")
print(null_summary[null_summary['결측치'] > 0].to_string())

# ── 2. purchase_option_color 결측 원인 분해 ──
po_null   = df['purchase_option'].isna()
col_null  = df['purchase_option_color'].isna()
size_null = df['purchase_option_size'].isna()

color_null_po_also_null  = (col_null & po_null).sum()     # purchase_option 자체가 null
color_null_po_has_value  = (col_null & ~po_null).sum()    # purchase_option 있는데 색상 없음

size_null_po_also_null   = (size_null & po_null).sum()
size_null_po_has_value   = (size_null & ~po_null).sum()

print("\n=== purchase_option_color 결측 원인 분해 ===")
print(f"  전체 결측: {col_null.sum():,}건")
print(f"  ① purchase_option 자체 null (데이터 없음)  : {color_null_po_also_null:,}건")
print(f"  ② purchase_option 있으나 색상 정보 없는 패턴: {color_null_po_has_value:,}건")

print("\n=== purchase_option_size 결측 원인 분해 ===")
print(f"  전체 결측: {size_null.sum():,}건")
print(f"  ① purchase_option 자체 null (데이터 없음)  : {size_null_po_also_null:,}건")
print(f"  ② purchase_option 있으나 사이즈 정보 없는 패턴: {size_null_po_has_value:,}건")

# ── 3. ②케이스의 purchase_option 패턴 분포 ──
print("\n=== 색상 null이지만 purchase_option 있는 케이스의 패턴 분포 ===")
color_null_with_po = df[col_null & ~po_null]['purchase_option'].astype(str)
print(color_null_with_po.apply(classify_pattern).value_counts().to_string())

print("\n=== 사이즈 null이지만 purchase_option 있는 케이스의 패턴 분포 ===")
size_null_with_po = df[size_null & ~po_null]['purchase_option'].astype(str)
print(size_null_with_po.apply(classify_pattern).value_counts().to_string())

# ── 4. user_height / user_weight 결측 ──
print("\n=== user_height / user_weight 결측 ──")
print(f"  user_height 결측: {df['user_height'].isna().sum():,}건 ({df['user_height'].isna().mean()*100:.1f}%)  → 회원이 신체정보 미입력")
print(f"  user_weight 결측: {df['user_weight'].isna().sum():,}건 ({df['user_weight'].isna().mean()*100:.1f}%)  → 회원이 신체정보 미입력")

height_only_null = (df['user_height'].isna() & df['user_weight'].notna()).sum()
weight_only_null = (df['user_weight'].isna() & df['user_height'].notna()).sum()
both_null        = (df['user_height'].isna() & df['user_weight'].isna()).sum()
print(f"  키만 없음: {height_only_null:,}건 / 몸무게만 없음: {weight_only_null:,}건 / 둘 다 없음: {both_null:,}건")

=== 컬럼별 결측치 요약 ===
                           전체  non-null     결측치  결측률(%)
purchase_option        494159    460810   33349    6.75
purchase_option_color  494159    351211  142948   28.93
purchase_option_size   494159    378727  115432   23.36
user_height            494159    433235   60924   12.33
user_weight            494159    425332   68827   13.93
user_height_group      494159    433235   60924   12.33
user_weight_group      494159    425332   68827   13.93

=== purchase_option_color 결측 원인 분해 ===
  전체 결측: 142,948건
  ① purchase_option 자체 null (데이터 없음)  : 33,349건
  ② purchase_option 있으나 색상 정보 없는 패턴: 109,599건

=== purchase_option_size 결측 원인 분해 ===
  전체 결측: 115,432건
  ① purchase_option 자체 null (데이터 없음)  : 33,349건
  ② purchase_option 있으나 사이즈 정보 없는 패턴: 82,083건

=== 색상 null이지만 purchase_option 있는 케이스의 패턴 분포 ===
purchase_option
사이즈만 KV    104585
옵션 단독        3181
추가상품         1819
기타             14

=== 사이즈 null이지만 purchase_option 있는 케이스의 패턴 분포 ===
purchase_option
색상만 KV    64893
선택N 패턴   

### 결측치 확인 결과
##### 색상/사이즈 결측 -> 애초에 정보를 제공하지 않음
##### 신체 정보 결측 -> 고객이 미입력한 정보

In [43]:
# 총 고유 컬러 수
all_colors = df['purchase_option_color'].dropna().str.split(',').explode().str.strip()
all_colors = all_colors[all_colors != '']

vc = all_colors.value_counts()
print(f"총 고유 컬러 수: {vc.shape[0]:,}개\n")

총 고유 컬러 수: 13,143개



In [44]:
color_df = vc.reset_index()
color_df.columns = ['컬러명', '건수']
color_df['비율(%)'] = (color_df['건수'] / len(all_colors) * 100).round(2)
color_df.index += 1

print(f"총 고유 컬러 수: {len(color_df):,}개")
color_df

총 고유 컬러 수: 13,143개


,컬러명,건수,비율(%)
1,['블랙'],86734,24.70
2,['화이트'],20026,5.70
3,"['블랙', '블랙']",13559,3.86
4,['백아이보리'],6522,1.86
5,['스킨'],4836,1.38
...,...,...,...
13139,"['문릿차콜', '아이보리']",1,0.00
13140,"['베일리핑크', '아이보리']",1,0.00
13141,['테디브라운'],1,0.00
13142,['코코넛크림'],1,0.00


In [45]:
# 1. 완벽한 문자열(string)로 변환 후 분리 및 평탄화
colors_str = df['purchase_option_color'].dropna().astype(str)
all_colors = colors_str.str.split(',').explode().str.strip()

# 2. 빈 문자열 제거
all_colors = all_colors[all_colors != '']

# 3. 고유 컬러 수 확인 (nunique() 사용)
unique_count = all_colors.nunique()
print(f"총 고유 컬러 수: {unique_count:,}개\n")

# 4. 상위/하위 10개 컬러 빈도수 확인
vc = all_colors.value_counts()
print("[가장 많이 구매된 컬러 Top 10]")
print(vc.head(10))
print("\n[가장 적게 구매된 컬러 Bottom 10]")
print(vc.tail(10))

총 고유 컬러 수: 1,637개

[가장 많이 구매된 컬러 Top 10]
purchase_option_color
블랙        191109
화이트        31893
백아이보리      18147
아이보리        8966
스킨          6897
멜란지그레이      6516
포그스킨        5735
네이비         5734
파우더크림       5261
차콜          5013
Name: count, dtype: int64

[가장 적게 구매된 컬러 Bottom 10]
purchase_option_color
네온팝핑핑크     1
아쿠아딥레드     1
바로크로즈      1
모스나이트XL    1
샌달우드       1
록시핑크       1
홀리핑크       1
더스트블루      1
멜트네이비      1
테디브라운      1
Name: count, dtype: int64


In [47]:
# 영문 알파벳(A-Z, a-z)이 하나라도 포함된 컬러명을 모두 추출 (가장 확실한 방법)
suspicious_colors = df[
    df['purchase_option_color'].notna() &
    df['purchase_option_color'].str.contains(r'[A-Za-z]', regex=True)
][['purchase_option', 'purchase_option_color', 'purchase_option_size']].drop_duplicates()

print(f"의심 컬러 행 수: {len(suspicious_colors):,}건\n")
if len(suspicious_colors) > 0:
    print(suspicious_colors.head(20).to_string())
else:
    print("🎉 영문 사이즈 찌꺼기가 포함된 컬러가 없습니다! 완벽합니다.")

의심 컬러 행 수: 525건

                                          purchase_option              purchase_option_color purchase_option_size
149024                              선택1:블랙 L / 선택2:로드네이비L                          로드네이비L,블랙                    L
174287                       선택1:더스틴딥그레이XL / 선택2:더스틴그린 XL                    더스틴그린,더스틴딥그레이XL                   XL
174299                    선택1:더스틴딥그레이XXL / 선택2:더스틴베이지 XXL                  더스틴딥그레이XXL,더스틴베이지                  XXL
174874                  선택1:더스틴딥그레이XXXL / 선택2:더스틴네이비 XXXL                 더스틴네이비,더스틴딥그레이XXXL                 XXXL
175078                     선택1:더스틴딥그레이XXL / 선택2:더스틴카키 XXL                   더스틴딥그레이XXL,더스틴카키                  XXL
176139                       선택1:더스틴딥그레이XL / 선택2:더스틴카키 XL                    더스틴딥그레이XL,더스틴카키                   XL
176538                          선택1:블랙 XL / 선택2:더스틴딥그레이XL                       더스틴딥그레이XL,블랙                   XL
247416  선택1:다크그레이M / 선택2:달리아와인M(+16395) / 선택3:블랙M(+16395)  다크그레이M,달리아와인

In [48]:
def clean_suspicious_options(row):
    c_val = row['purchase_option_color']
    s_val = row['purchase_option_size']
    
    # 1. 색상 결측치 안전하게 검사 (float NaN 방어)
    if pd.isna(c_val) or str(c_val).strip() == 'nan':
        return pd.Series([pd.NA, pd.NA if pd.isna(s_val) or str(s_val).strip() == 'nan' else s_val])
        
    # 강제 문자열 변환 후 split 적용
    c_list = [c.strip() for c in str(c_val).split(',')]
    new_c_list = []
    new_s_list = []
    
    extracted_size = False
    
    for c in c_list:
        c = re.sub(r'\(\+[0-9]+\)', '', c).strip()
        
        m = re.search(r'(?i)(XL|XXL|XXXL|XS|S|M|L|F|FREE|ONE\s*SIZE)$', c)
        if m:
            new_s_list.append(m.group(1).upper()) 
            c = c[:m.start()].strip()             
            extracted_size = True
        else:
            new_s_list.append(pd.NA)
            
        m_kor = re.findall(r'[가-힣]+', c)
        if m_kor:
            clean_w = [w for w in m_kor if w not in ['선택', '추가', '혜택', '추가안함']]
            new_c_list.append(clean_w[-1] if clean_w else pd.NA) 
        else:
            new_c_list.append(pd.NA)
            
    c_final = ', '.join([str(x) for x in new_c_list if pd.notna(x)])
    
    # 2. 사이즈 결측치 안전하게 검사 및 split (float 에러 해결 부분)
    is_s_null = pd.isna(s_val) or str(s_val).strip() == 'nan'
    s_orig_list = [s.strip() for s in str(s_val).split(',')] if not is_s_null else []
    
    if is_s_null or (extracted_size and len(s_orig_list) < len(new_c_list)):
        s_final = ', '.join([str(x) for x in new_s_list if pd.notna(x)])
    else:
        s_final = s_val if not is_s_null else pd.NA
        
    return pd.Series([c_final if c_final else pd.NA, s_final if s_final else pd.NA])


# ── 실행 로직 ──
mask = df['purchase_option_color'].astype(str).str.contains(r'[A-Za-z]|\(\+', regex=True, na=False)
print(f"🛠 후처리 타겟 행 수: {mask.sum():,}건")

df.loc[mask, ['purchase_option_color', 'purchase_option_size']] = df[mask].apply(clean_suspicious_options, axis=1)

# 카테고리 타입으로 원상 복구
df['purchase_option_color'] = df['purchase_option_color'].astype('category')
df['purchase_option_size'] = df['purchase_option_size'].astype('category')

print("✓ 찌꺼기 후처리 완료!")

🛠 후처리 타겟 행 수: 1,050건
✓ 찌꺼기 후처리 완료!


In [49]:
# 영문 알파벳(A-Z, a-z)이 하나라도 포함된 컬러명을 모두 추출 (가장 확실한 방법)
suspicious_colors = df[
    df['purchase_option_color'].notna() &
    df['purchase_option_color'].str.contains(r'[A-Za-z]', regex=True)
][['purchase_option', 'purchase_option_color', 'purchase_option_size']].drop_duplicates()

print(f"의심 컬러 행 수: {len(suspicious_colors):,}건\n")
if len(suspicious_colors) > 0:
    print(suspicious_colors.head(20).to_string())
else:
    print("🎉 영문 사이즈 찌꺼기가 포함된 컬러가 없습니다! 완벽합니다.")

의심 컬러 행 수: 0건

🎉 영문 사이즈 찌꺼기가 포함된 컬러가 없습니다! 완벽합니다.


In [50]:
# 1. 완벽한 문자열(string)로 변환 후 분리 및 평탄화
colors_str = df['purchase_option_color'].dropna().astype(str)
all_colors = colors_str.str.split(',').explode().str.strip()

# 2. 빈 문자열 제거
all_colors = all_colors[all_colors != '']

# 3. 고유 컬러 수 확인 (nunique() 사용)
unique_count = all_colors.nunique()
print(f"총 고유 컬러 수: {unique_count:,}개\n")

# 4. 상위/하위 10개 컬러 빈도수 확인
vc = all_colors.value_counts()
print("[가장 많이 구매된 컬러 Top 10]")
print(vc.head(10))
print("\n[가장 적게 구매된 컬러 Bottom 10]")
print(vc.tail(10))

총 고유 컬러 수: 1,456개

[가장 많이 구매된 컬러 Top 10]
purchase_option_color
블랙        191014
화이트        31893
백아이보리      18145
아이보리        8925
스킨          6897
멜란지그레이      6516
네이비         5734
포그스킨        5732
파우더크림       5260
차콜          5013
Name: count, dtype: int64

[가장 적게 구매된 컬러 Bottom 10]
purchase_option_color
모노로즈      1
네온팝핑핑크    1
아쿠아딥레드    1
바로크로즈     1
샌달우드      1
록시핑크      1
홀리핑크      1
더스트블루     1
멜트네이비     1
테디브라운     1
Name: count, dtype: int64


In [ ]:
# 데이터 저장
# df.to_csv('./final_data/xexymix_homepage_review_final_s2024.csv', index=False)